|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The KV cache<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: write both loops and make them agree<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# The naive loop runs a forward pass at every sequence length from 1 to N,
# so it asks the allocator for N different block sizes. The default
# allocator caches every one of them and then runs out of room on a card
# with 12 GB free. This mode grows one segment instead. Set it BEFORE
# torch is imported, or it does nothing.
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import cudalib

Two generation loops that must produce the same text.

One recomputes the whole prefix every step. The other keeps K and V and feeds
the model one token at a time. Write both, prove they agree, then find out
what the cache actually bought you.

This is stage 02 of the ladder, in miniature, using HuggingFace's cache
rather than one you built.

In [ ]:
### run this cell

MODEL = 'Qwen/Qwen3-0.6B'
tok   = AutoTokenizer.from_pretrained(MODEL)

def load(dtype):
  return AutoModelForCausalLM.from_pretrained(MODEL, dtype=dtype).cuda().eval()

model  = load(torch.bfloat16)
prompt = tok('The capital of France is', return_tensors='pt').input_ids.cuda()
print(tok.decode(prompt[0]))

# Exercise 1: the naive loop

Greedy decoding, no cache. The whole sequence goes through the model on every
single step.

In [ ]:
@torch.inference_mode()
def naive_generate(model, ids, n):
  x = ids.clone()
  for _ in range(n):
    logits = model(x, use_cache=False).logits
    nxt    = logits[:, -1:].argmax(-1)
    x      = torch.cat([x, nxt], dim=1)
  return x

out = naive_generate(model, prompt, 20)
print(tok.decode(out[0]))

# Exercise 2: the cached loop

The prompt goes through once. After that each step sees one token, and the
model reads the rest out of `past_key_values`.

The trap is in what you feed it on the second pass. Feed the whole sequence
again and the cache is silently appended to a prefix it already holds.

In [ ]:
@torch.inference_mode()
def cached_generate(model, ids, n):
  # the prompt goes through once, in full. This is prefill.
  out  = model(ids, use_cache=True)
  past = out.past_key_values
  nxt  = out.logits[:, -1:].argmax(-1)
  x    = torch.cat([ids, nxt], dim=1)

  # after that, one token at a time. This is decode.
  for _ in range(n-1):
    out  = model(nxt, past_key_values=past, use_cache=True)
    past = out.past_key_values
    nxt  = out.logits[:, -1:].argmax(-1)
    x    = torch.cat([x, nxt], dim=1)
  return x

out = cached_generate(model, prompt, 20)
print(tok.decode(out[0]))

# Exercise 3: do they agree?

Same model, same prompt, same greedy rule. The two loops should write the
same sentence.

In [ ]:
a = naive_generate(model, prompt, 24)
b = cached_generate(model, prompt, 24)

print('identical:', torch.equal(a, b))
print('\nnaive :', tok.decode(a[0]))
print('cached:', tok.decode(b[0]))

diff = next((i for i,(u,v) in enumerate(zip(a[0], b[0])) if u != v), None)
print(f'\nfirst token that differs: {diff}')

# Exercise 4: what did the cache buy?

Time both loops at a few lengths. One of them is linear in the number of
tokens and the other is not, so the gap should widen as you generate more.

The first cell of this notebook sets an allocator option, and it is not
superstition. The naive loop runs a forward pass at 512 different sequence
lengths, so it asks the allocator for 512 different block sizes and the
default allocator caches every one of them. Without that option the notebook
reports an out-of-memory warning on a card with 12 GB free.

Every new shape has a cost. You will meet that fact twice more: on the JAX
track it is a recompile per shape, and in stage 12 it is why CUDA graphs are
captured at bucketed batch sizes.

In [ ]:
import gc

for n in (32, 128, 512):
  gc.collect(); torch.cuda.empty_cache()
  ms_naive  = cudalib.bench_ms(lambda: naive_generate(model, prompt, n),
                               iters=1, warmup=1)
  ms_cached = cudalib.bench_ms(lambda: cached_generate(model, prompt, n),
                               iters=1, warmup=1)
  print(f'{n:>4} tokens: naive {ms_naive/1000:6.2f} s   '
        f'cached {ms_cached/1000:5.2f} s   {ms_naive/ms_cached:5.2f}x')

# Exercise 5: now try Exercise 3 again in fp32

Same two loops, same prompt, one thing changed. This runs last because fp32
weights are twice the size and crowd the card.

In [ ]:
exact  = load(torch.float32)

a = naive_generate(exact, prompt, 24)
b = cached_generate(exact, prompt, 24)
print('fp32 identical:', torch.equal(a, b))

# fp32 is twice the weights. Give the memory back before the timings,
# or the next cell fragments the allocator and thrashes.
import gc
del exact, a, b
gc.collect()
torch.cuda.empty_cache()

### The two results, and neither is a bug

**In bf16 the two loops write different sentences.** They agree for a
while and then split, usually around the tenth token. In fp32 they are
identical.

Nothing is broken. The cached path reduces in a different order from the
uncached one, so logits land about 1e-2 apart. Wherever the top two
candidates are close, that is enough to flip an argmax, and one different
token changes every token after it.

This is a real property of low-precision inference, not an artefact of
this exercise. It is why production LLM serving is not reproducible
across batch sizes or cache configurations, and it is why every
correctness check in this repo that compares two implementations runs in
fp32. Look at the `hf_exact` fixture in `tests/conftest.py`.

**The speedup is smaller than the complexity argument promised.** At 32
tokens the cache may even lose. A 0.6B model spends most of a decode step
in Python and kernel launches, and the cache does nothing about those.
The win grows with the number of tokens, because one side is linear and
the other is not, and it grows with model size, because the fixed tax
stays fixed.

Now go and build this properly, with the cache as an object you own:

    ./vc guide 2